In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

NAME = "wolf4"

D = next(
    r for r, _, f in os.walk("/kaggle/input")
    if "train.csv" in f
)

MODEL = next(
    r for r, _, f in os.walk("/kaggle/input")
    if "config.json" in f and "wolf" in r.lower()
)

print(D)
print(MODEL)

train = pd.read_csv(f"{D}/train.csv")
test = pd.read_csv(f"{D}/test.csv")

labels = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
l2i = {label: i for i, label in enumerate(labels)}

train["y"] = train["label"].map(l2i)

print(train.groupby(["language", "label"]).size().unstack())

/kaggle/input/competitions/rmit-hackathon-2026
/kaggle/input/models/aisuko/wolf-defender-prompt-injection/transformers/default/1
label     BENIGN  JAILBREAK  PROMPT_INJECTION
language                                     
en           868        434                98
fr          2170       1085               245
it          1736        868               196
pt          3906       1953               441


In [2]:
MAXLEN = 256
tok = AutoTokenizer.from_pretrained(MODEL)

class DS(Dataset):
    def __init__(self, texts, labels_y=None):
        self.e = tok(list(texts), truncation=True, max_length=MAXLEN)
        self.y = labels_y

    def __len__(self):
        return len(self.e["input_ids"])

    def __getitem__(self, i):
        item = {key: value[i] for key, value in self.e.items()}
        if self.y is not None:
            item["labels"] = int(self.y[i])
        return item

strat = train["language"] + "_" + train["label"]

tr, va = train_test_split(
    train,
    test_size=0.1,
    stratify=strat,
    random_state=42,
)

cnt = np.bincount(tr["y"], minlength=3)
w = torch.tensor((len(tr) / (3 * cnt)) ** 0.5, dtype=torch.float)

class WT(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        y = inputs.pop("labels")
        out = model(**inputs)
        loss = nn.CrossEntropyLoss(
            weight=w.to(out.logits.device)
        )(out.logits, y)

        return (loss, out) if return_outputs else loss

def metrics(prediction):
    return {
        "macro_f1": f1_score(
            prediction.label_ids,
            prediction.predictions.argmax(1),
            average="macro",
        )
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
)

args = TrainingArguments(
    "out_wolf4",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=64,
    learning_rate=3e-5,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

t = WT(
    model=model,
    args=args,
    train_dataset=DS(tr["text"], tr["y"].values),
    eval_dataset=DS(va["text"], va["y"].values),
    data_collator=DataCollatorWithPadding(tok),
    compute_metrics=metrics,
)

t.train()

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: /kaggle/input/models/aisuko/wolf-defender-prompt-injection/transformers/default/1
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.127164,0.946800
2,No log,0.169654,0.960082


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=394, training_loss=0.16267832644699795, metrics={'train_runtime': 513.0939, 'train_samples_per_second': 49.114, 'train_steps_per_second': 0.768, 'total_flos': 2146789312819200.0, 'train_loss': 0.16267832644699795, 'epoch': 2.0})

In [3]:
pv = t.predict(DS(va["text"], va["y"].values)).predictions

print(classification_report(
    va["y"],
    pv.argmax(1),
    target_names=labels,
))

for language, group in va.assign(prediction=pv.argmax(1)).groupby("language"):
    print(
        language,
        f1_score(group["y"], group["prediction"], average="macro"),
    )

best = (
    f1_score(va["y"], pv.argmax(1), average="macro"),
    np.zeros(3),
)

for b1 in np.arange(-1, 1.01, 0.25):
    for b2 in np.arange(-1, 2.01, 0.25):
        bias = np.array([0, b1, b2])
        score = f1_score(
            va["y"],
            (pv + bias).argmax(1),
            average="macro",
        )
        if score > best[0] + 1e-3:
            best = (score, bias)

print("Best calibrated Macro F1 and bias:", best)

                  precision    recall  f1-score   support

          BENIGN       0.98      0.97      0.97       869
       JAILBREAK       0.95      0.96      0.95       433
PROMPT_INJECTION       0.96      0.95      0.95        98

        accuracy                           0.97      1400
       macro avg       0.96      0.96      0.96      1400
    weighted avg       0.97      0.97      0.97      1400

en 0.9277660324171952
fr 0.9875402286207337
it 0.9728159736517085
pt 0.947991228847454
(0.9647113691389735, array([ 0.  , -1.  , -0.75]))


In [5]:
pt = t.predict(DS(test["text"])).predictions

submission = pd.DataFrame({
    "id": test["id"],
    "label": [labels[i] for i in (pt + best[1]).argmax(1)],
})

submission.to_csv("submission_wolf4.csv", index=False)

print(submission.shape)
print(submission["label"].value_counts(normalize=True))

np.save(f"/kaggle/working/val_{NAME}.npy", pv)
np.save(f"/kaggle/working/test_{NAME}.npy", pt)
np.save("/kaggle/working/val_y.npy", va["y"].values)

print(os.listdir("/kaggle/working"))

(6000, 2) label
BENIGN              0.6260
JAILBREAK           0.3035
PROMPT_INJECTION    0.0705
Name: proportion, dtype: float64
['val_y.npy', '.virtual_documents', 'submission.csv', 'val_wolf.npy', 'out', 'test_wolf.npy']
